In [ ]:
# Celda 1: Entorno y carga
import runpy
from pathlib import Path
from pyspark.sql import functions as F

runpy.run_path(str(next(p / "notebooks" / "_init_env.py" for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "notebooks" / "_init_env.py").is_file())))

from src.catalog import SILVER_AEMET_TABLE

df_aemet = spark.table(SILVER_AEMET_TABLE)

In [ ]:
# Celda 2: Checks de calidad (claves, rangos y unicidad)
assert df_aemet.count() > 0, "SILVER_AEMET_TABLE está vacía"
assert df_aemet.filter(F.col("indicativo").isNull() | F.col("fecha_dt").isNull()).count() == 0, "Claves nulas"
assert df_aemet.filter((F.col("hr_media_pct") < 0) | (F.col("hr_media_pct") > 100)).count() == 0, "HR fuera de [0, 100]"
assert df_aemet.count() == df_aemet.dropDuplicates(["indicativo", "fecha_dt"]).count(), "Duplicados estación-día"
print("Checks AEMET OK. Total filas:", df_aemet.count())

In [ ]:
# Celda 3: Top eventos de sequía y estrés termohigrométrico
display(
    df_aemet.orderBy(
        F.col("dias_secos_consecutivos").desc(),
        F.col("indice_sequedad_termohigro").desc_nulls_last(),
    )
    .select(
        "fecha_dt",
        "provincia",
        "nombre_estacion",
        "dias_secos_consecutivos",
        "prec_acum_30d_mm",
        "tmax_c",
        "hr_min_pct",
        "indice_sequedad_termohigro",
    )
    .limit(15)
)

In [ ]:
# Celda 4: Perfil estacional por provincia y mes
display(
    df_aemet.groupBy("provincia", F.month("fecha_dt").alias("mes"))
    .agg(
        F.round(F.avg("dias_secos_consecutivos"), 1).alias("racha_seca_media"),
        F.round(F.avg("indice_sequedad_termohigro"), 2).alias("sequedad_media"),
        F.round(F.avg("prec_acum_30d_mm"), 1).alias("prec_30d_media"),
    )
    .orderBy("provincia", "mes")
)